# Generacion Sabana Analitica Propension Compras

## PASOS PREVIOS

### Carga de Librerias

In [1]:
import platform
from hdbcli import dbapi
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta
import configparser
import sqlalchemy
import os

### Funciones Auxiliares

In [2]:
def read_sql_file(filepath):
   with open(filepath, 'r', encoding='utf-8') as file:
       return file.read()

In [3]:
def conexion_hana(credenciales):
    ## credenciales
    config_file=credenciales
    config=configparser.ConfigParser()
    config.read(config_file)
    ## conexion
    #datos conexion
    conn = dbapi.connect(address=config['HANA']['address'],
    port=config['HANA']['port'],
    user=config['HANA']['user'],
    password=config['HANA']['password'])
    return (conn)

In [4]:
def guardar_df(df, ruta):
    dia=datetime.now().day
    mes=datetime.now().strftime('%b')
    #mes_corte=(datetime.now()-relativedelta(months=1)).strftime('%b%y')
    mes_corte='Jul26'
    anio=datetime.now().year
    nombre_archivo = f'InsumoHana_SabanaAnalitica_PropCompras_{dia:02d}{mes}{anio}_{mes_corte}.parquet'        
    ruta_completa = os.path.join(ruta, nombre_archivo)
    df.to_parquet(ruta_completa)

## Extraccion

#### conexion Hana

In [5]:
#conn=conexion_hana('D:/Users/mney/OneDrive - Compartamos Banco/ENTREGA CESAR/config.ini')
conn=conexion_hana('D:/Users/magrios/OneDrive - Compartamos Banco/Modelos/NivelTransaccional (Churn)/config.ini')

#### Asignacion Fecha de ejecucion. <br>
Se considera el primer dia del mes previo a la ejecucion, por ejemplo <br>
Si se ejecuta  el 02 de junio del 2025 se tomara en consideracion el 01 de mayor del 2025, esto para poder capturar las cuentas activas al mes cerrado anterior


fecha=(datetime.now().replace(day=1)-relativedelta(months=1)).strftime("%Y-%m-%d")

fecha='2025-01-01'

#### Ubicacion del query

filepath = "D:/Users/cjvelazquez/SEGMENTACION_CLIENTES_AHORRO/QueryHana_SegCliAho_Prediccion_19jun25_v2.sql"
sql = read_sql_file(filepath)

### Lectura Hana


In [6]:
sql='''
SELECT * FROM Sabana_Analitica_PropCompras
'''

In [7]:
chunks = pd.read_sql(sql, conn, chunksize=10000)

D:\Users\magrios\AppData\Local\Temp\ipykernel_23836\2578911268.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  chunks = pd.read_sql(sql, conn, chunksize=10000)


In [8]:
%%time
df_final = pd.concat(chunks, ignore_index=True)

CPU times: total: 6min 14s
Wall time: 2h 29min 6s


In [9]:
df_final.head(3)

,NUM_CLIENTE,BP,NUM_CUENTA_AHORRO,ID_ESTATUS_CUENTA,ANTIGUEDAD_AHORRO,CUENTA_AHORRO,FEC_APERTURA_CUENTA,ESTATUS_CUENTA_DESC,ID_PRODUCTO_AHORRO,PRODUCTO_AHORRO,...,CANAL_DISPERSION,ID_METODO_DISPERSION,METODO_DISPERSION,ID_DIRECCION_REGIONAL,DIRECCION_REGIONAL,COORDENADA_XOS,COORDENADA_YOS,OFICINA_SERVICIO,CENTRO_COSTOS,FLAG_CRD_HIST
0,105741427,91833417,105229561,3,3,00207876217,2026-04-03,"ACTIVOS, UTILIZADOS",64,CUENTA NIVEL 2 EXPRESS,...,None,NaN,None,22659092,OPERACIONES SUCURSALES,19.368612,-99.180237,30101001,1230101001,-1
1,107079951,92216561,106973975,3,2,00209645878,2026-05-27,"ACTIVOS, UTILIZADOS",64,CUENTA NIVEL 2 EXPRESS,...,None,NaN,None,22659092,OPERACIONES SUCURSALES,19.368612,-99.180237,30101001,1230101001,-1
2,106982783,92190600,106865529,3,2,00209536139,2026-05-25,"ACTIVOS, UTILIZADOS",64,CUENTA NIVEL 2 EXPRESS,...,None,NaN,None,22659092,OPERACIONES SUCURSALES,19.368612,-99.180237,30101001,1230101001,-1


In [10]:
df_final.shape

(2903623, 242)

In [11]:
df_final.shape

(2903623, 242)

#### Lectura Hana

%%time
df=pd.read_sql_query(sql.format(fecha=fecha),conn)

%%time
df=pd.read_sql_query(sql,conn)

## Almacenado

#### Visualizacion

df.head(3)

df.shape

sql='''
DROP table Sabana_Analitica_PropCompras
'''

df=pd.read_sql_query(sql,conn)

cursor = conn.cursor()
cursor.execute("DROP TABLE Sabana_Analitica_PropCompras")
conn.commit()

In [14]:
#ruta='D:\Users\mney\OneDrive - Compartamos Banco\ENTREGA CESAR\Propension Compras\sabana'
ruta='D:/Users/magrios/OneDrive - Compartamos Banco/Modelos'

In [15]:
guardar_df(df_final, ruta)

### ELIMINAR TABLA

In [16]:
cursor = conn.cursor()
sql_drop = f"DROP table Sabana_Analitica_PropCompras;"
cursor.execute(sql_drop)
conn.commit()

In [17]:

# cerrar conexion
conn.close()